# Solar Active-Region Detection — Kaggle training run (3-channel MAX-QUALITY)

**One cell to run** (safe to re-run any time: stops the old run, re-clones the
small code repo, resumes data + training from where they stopped).

Right-hand panel (**Session options**):

1. **Internet: ON** — required for the download.
2. **GPU: T4 x2** — both are used automatically (DataParallel).
3. **Persistence: Files only** — so each 12-hour session end KEEPS your data + model.

## Why 3 channels / 550 frames

Kaggle's persistent working+output disk is hard-capped at **20 GiB** (Kaggle staff
confirmed). A 13-channel tiled frame is ~90 MB, so the Mac's 2,000-frame dataset
(≈180 GiB) cannot live here. This preset uses the code's own recommended strong
from-scratch trio — **AIA 171 + AIA 193 + magnetogram** ("active regions are
defined by the field") — which is ¼ the size: 550 frames (≈4,500 tiles) peaks
at ~19 GiB, under the wall, and persists across sessions.

Preset: `BASE_CHANNELS=48 DEEP_SUPERVISION=1` (best-quality 18M model, batch
auto-fits VRAM), 6 download workers, every core, both T4s.

When a session ends (12 h): run this same cell again in the same notebook — it
resumes. The newest model is mirrored to /kaggle/output (Output tab) every 5 min.

In [ ]:
%%bashset -xexport HOME=/kaggle/workingcd /kaggle/working# 1) Stop any previous run (lock PID + trainer + downloader), and clean orphaned temp files:P=$(cat /kaggle/working/solar_results/arpil/run_forever.lock 2>/dev/null)[ -n "$P" ] && kill -TERM "$P" 2>/dev/nullpkill -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullsleep 5pkill -9 -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -9 -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullrm -rf /tmp/arpil_resume_* 2>/dev/null# 2) One-time switch from the 13-channel attempt to the 3-channel preset:#    the old tiles/checkpoints use a different channel set. Marker file makes#    this happen ONCE; all later re-runs (12-h resumes) keep data + model.if [ ! -f /kaggle/working/.solar3ch ]; then    rm -rf /kaggle/working/solar_data /kaggle/working/solar_results    touch /kaggle/working/.solar3chfi# 3) Fresh code repo (tiny, ~15 s):rm -rf /kaggle/work SOALRgit clone -q -b arena/01a04247-soalr-active-region-detection \    https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \|| { mkdir -p SOALR && wget -qO /tmp/repo.tgz \    https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \    && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }if [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# 4) NO venv on Kaggle (ensurepip is broken there); torch is preinstalled:python3 -m pip install -q -r requirements.txtpython3 -m pip cache purge 2>/dev/nullpython3 -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"# 5) Mirror the newest model + log to /kaggle/output every 5 min (downloadable#    from the Output tab after a session ends):mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/working/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# 6) 3-CHANNEL MAX-QUALITY preset sized to stay under Kaggle's 20 GiB working#    quota: 550 frames x ~21 MB tiles + 6 in-flight downloads + archive#    peaks at ~19 GiB - under the wall. Both T4s used automatically.CHANNELS="aia171 aia193 hmi_m" \SOLAR_PYTHON="$(command -v python3)" BASE_CHANNELS=48 DEEP_SUPERVISION=1 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=550 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=6 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- `torch 2.x.x | CUDA available: True | GPUs: 2` — environment healthy
- Preflight `3 ok`, then `Downloading 200 frames with 6 parallel workers ...`
- `[stream] using 2 GPUs with DataParallel (each batch splits across both)`
- `[stream] epoch=0001 loss=0.7x ...` — finite loss; first `val_dice` at epoch 10

## Disk

The working+output disk is capped at 20 GiB by Kaggle. This preset peaks at
~19 GiB (550 frames of 3-channel tiles + in-flight downloads + mask archive)
and plateaus there — the Output number should stop growing once the 550-frame
cap is reached.

## Re-running

This cell is idempotent: it stops any running instance, re-clones the code
(~15 s), and resumes. With **Persistence: Files only**, all downloaded data and
checkpoints survive — only the code is refreshed.